In [1]:
import numpy as np
import cvxpy as cp
import qutip as qt

In [2]:
def createKraus(dims=[5,5], params=[.1,.1],method = 'operator'):
    """Generate a channel that represents mixing the first mode (signal) with a thermal state and return the Kraus rep
    ----------
    dims: int[3], default [5,5]
        The size of the Fock spaces for respectively, the thermal modes, and the signal mode
    params: float[2], default [.1,.1]
        The transmisivity of the beamsplitter and the average photon number of the thermal field
    method: string literal, default 'operator'
        Either 'operator' or 'analytic', determines the method used to generate the thermal state as seen in qutip's thermal_dm method
    Returns
    -------
    QuTiP list[Qobj] (oper)
        The Kraus representation of a superoperator that acts on states to apply the above defined channel
    """
    eta = params[0]
    theta = np.arccos(np.sqrt(eta))
    nth = params[1]
    thdims = dims[0]
    sdims = dims[1]
    # Create our transformation, and our state to compare with
    ath = qt.tensor(qt.destroy(thdims),qt.identity(sdims))
    asi = qt.tensor(qt.identity(thdims),qt.destroy(sdims)) # can't call this as, because it's a keyword
    G = theta *1j*(ath.dag()*asi +asi.dag()*ath)
    # Define the beamsplitter transform
    U = G.expm()
    pth = qt.thermal_dm(thdims,nth,method)
    # Create a superoperator which performs the thermal mixing + tracing out
    S1=0
    for i in range(thdims):
        fockn = qt.fock(thdims,i)
        pthket = pth*fockn
        ql = qt.tensor(qt.identity(sdims),pthket).drop_scalar_dims(inplace=True)
        qr = qt.tensor(qt.identity(sdims),fockn.dag()).drop_scalar_dims(inplace=True)
        S1 = S1+qt.sprepost(ql,qr)
    S2 = qt.sprepost(U,U.dag())*S1
    S3 = qt.tensor_contract(S2,(1,3))
    krausqm = qt.to_kraus(S3)
    krausms = [qt.dimensions.to_tensor_rep(krausqm[i]) for i in range(len(krausqm))]
    are_real = np.all(np.isreal(krausms))
    if are_real:
        return np.real(krausms)
    else:
        return krausms

In [ ]:
def channelCapacity(dim,eta,nth):
    Kdim = dim**2
    Kraus = createKraus(dims=[dim,dim],params=[eta,nth])
    num = [ np.insert(np.zeros(dim-1),i,i) for i in range(dim)]
    Krausd = np.matmul(Kraus,num)
    Kraust = np.transpose(Kraus,[0,2,1])
    Krausdt = np.transpose(Krausd,[0,2,1])
    H = cp.Variable((Kdim,Kdim),hermitian=True)
    lam = cp.Variable()

In [3]:
dim = 4
Kdim = dim**2
eta = .1
nth = .1

In [4]:
Kraus = createKraus(dims=[dim,dim],params=[eta,nth])

In [5]:
num = [ np.insert(np.zeros(dim-1),i,i) for i in range(dim)]

In [6]:
Krausd = np.matmul(Kraus,num)

In [7]:
Kraust = np.transpose(Kraus,[0,2,1])

In [8]:
Krausdt = np.transpose(Krausd,[0,2,1])

In [9]:
#numb = [ np.insert(np.zeros(len(Kraus)-1),i,i) for i in range(len(Kraus))]

In [10]:
#Ktest = np.moveaxis(np.matmul(numb,np.moveaxis(Kraus,0,1)),1,0)

In [11]:
H = cp.Variable((Kdim,Kdim),hermitian=True)

In [12]:
lam = cp.Variable()

In [13]:
Ht = cp.conj(H)

In [14]:
Ktildep = cp.swapaxes(cp.matmul(H,np.swapaxes(Kraus,0,1)),1,0)

In [15]:
Ktilde = Krausd -1j*Ktildep

In [16]:
Ktildept = cp.swapaxes(cp.matmul(Ht,np.swapaxes(Kraust,0,1)),1,0)

In [17]:
Ktildedag = Krausdt + 1j*Ktildept

In [18]:
#prods = cp.matmul(Ktildedag,Ktilde)

In [19]:
#mat = cp.sum(prods,axis=0)

In [20]:
to_big = [[ 0 for i in range(Kdim+1)] for j in range(Kdim+1)]
for i in range(Kdim+1):
    for j in range(Kdim+1):
        if i==0:
            if j==0:
                to_big[i][j] = np.zeros((dim,dim))
            else:
                to_big[i][j] = Ktildedag[j-1]
        elif j==0:
            to_big[i][j] = Ktilde[i-1]
        else:
            to_big[i][j] = np.zeros((dim,dim))

In [21]:
base = cp.bmat(to_big)

In [22]:
lamdag = lam*np.identity(dim*(Kdim+1))

In [23]:
A = lamdag+base

In [24]:
A

Expression(AFFINE, UNKNOWN, (68, 68))

In [25]:
constraints = [A >> 0]

In [26]:
prob = cp.Problem(cp.Minimize(4*cp.abs(lam)**2),constraints)

In [27]:
#testprob.solve(verbose=True)

In [28]:
sol = prob.solve(verbose=True,solver=cp.SCS)

(CVXPY) Apr 24 12:48:48 PM: Your problem has 257 variables, 4624 constraints, and 0 parameters.
(CVXPY) Apr 24 12:48:48 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Apr 24 12:48:48 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Apr 24 12:48:48 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Apr 24 12:48:48 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Apr 24 12:48:48 PM: Compiling problem (target solver=SCS).
(CVXPY) Apr 24 12:48:48 PM: Reduction chain: Complex2Real -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Apr 24 12:48:48 PM: Applying reduction Complex2Real
(CVXPY) Apr 24 12:48:48 PM: Applying reduction Dcp2Cone
(CVXPY) Apr 24 12:48:48 PM: Applying reduction CvxAttr2Constr
(CVXPY) Apr 24 12:48:48 PM: Applying reduction ConeMatrixStuffing


                                     CVXPY                                     
                                     v1.8.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


/tmp/ipykernel_2014300/1406343072.py:1: UserWarning: The problem includes expressions that don't support CPP backend. Defaulting to the SCIPY backend for canonicalization.
  sol = prob.solve(verbose=True,solver=cp.SCS)
(CVXPY) Apr 24 12:48:49 PM: Applying reduction SCS
(CVXPY) Apr 24 12:48:49 PM: Finished problem compilation (took 3.811e-01 seconds).
(CVXPY) Apr 24 12:48:49 PM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.9 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 258, constraints m: 9318
cones: 	  l: linear vars: 2
	  s: psd vars: 9316, ssize: 1
settings: eps_abs: 1.0e-05, eps_rel: 1.0e-05, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 100000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 14648, nnz(P): 1
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |  scale  | time (s)
---

(CVXPY) Apr 24 12:48:49 PM: Problem status: optimal
(CVXPY) Apr 24 12:48:49 PM: Optimal value: 3.600e+01
(CVXPY) Apr 24 12:48:49 PM: Compilation took 3.811e-01 seconds
(CVXPY) Apr 24 12:48:49 PM: Solver (including time spent in interface) took 3.417e-01 seconds


   150| 6.34e-12  1.07e-11  7.18e-11  3.60e+01  1.00e-01  3.40e-01 
------------------------------------------------------------------
status:  solved
timings: total: 3.40e-01s = setup: 5.94e-03s + solve: 3.34e-01s
	 lin-sys: 1.60e-02s, cones: 3.11e-01s, accel: 2.55e-03s
------------------------------------------------------------------
objective = 36.000000
------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


In [29]:
sol

np.float64(35.99999999999107)